In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def fetch_full_content(article_info):
    """기사 본문에 접속하여 텍스트를 가져오고 정제합니다."""
    try:
        resp = session.get(article_info['링크'], headers=headers, timeout=7)
        resp.encoding = 'euckr'
        soup = BeautifulSoup(resp.text, 'html.parser')
        content_area = soup.select_one('#news_read')
        
        if content_area:
            # 본문 내 불필요 태그(이미지 설명, 관련뉴스 등) 제거
            for s in content_area.select('.link_news, .article_bottom, .promotion, script, style'): 
                s.decompose()
            article_info['본문'] = clean_text(content_area.get_text())
        else:
            article_info['본문'] = "본문 없음"
    except Exception:
        article_info['본문'] = "수집 실패"
    
    return article_info

def run_general_scraper():
    print(f"📅 [{TARGET_DATE}]부터 오늘까지의 모든 기사 수집을 시작합니다.")
    
    # --- [2단계: 기사 목록 전체 확보] ---
    all_list = []
    page = 1
    # 네이버 증권 '실시간 속보' 페이지
    base_url = "https://finance.naver.com/news/mainnews.naver?date=2024-10-25"
    
    while True:
        try:
            resp = session.get(base_url, params={'page': page}, headers=headers)
            resp.encoding = 'euckr'
            soup = BeautifulSoup(resp.text, 'html.parser')
            
            articles = soup.select('dl')
            if not articles: break
            
            finished = False
            for article in articles:
                sub_tag = article.select_one('.articleSubject a')
                wdate_tag = article.select_one('.wdate')
                press_tag = article.select_one('.press')
                
                # 유효한 뉴스 항목인지 확인 (에러 방지)
                if not sub_tag or not wdate_tag:
                    continue
                
                wdate_text = wdate_tag.get_text().strip()
                article_date = wdate_text.split(' ')[0]
                
                # 기간을 벗어나면 중단
                if article_date < TARGET_DATE:
                    finished = True
                    break
                    
                all_list.append({
                    '날짜': wdate_text,
                    '언론사': press_tag.get_text().strip() if press_tag else "알수없음",
                    '제목': sub_tag.get_text().strip(),
                    '링크': "https://finance.naver.com" + sub_tag['href']
                })
            
            if finished: break
            print(f"목록 확보 중... 현재 {page}페이지", end='\r')
            page += 1
            time.sleep(0.1)
            
        except Exception as e:
            print(f"\n목록 페이지 읽기 오류: {e}")
            break

    print(f"\n\n✅ 수집된 목록: 총 {len(all_list)}건")

    # --- [3단계: 본문 병렬 수집 (속도 극대화)] ---
    print(f"🚀 본문 수집 및 개인정보 정제 시작 (Thread: 10)...")
    with ThreadPoolExecutor(max_workers=10) as executor:
        final_results = list(executor.map(fetch_full_content, all_list))

    # --- [4단계: 데이터 저장] ---
    df = pd.DataFrame(final_results)
    if not df.empty:
        filename = f"naver_finance_all_{datetime.now().strftime('%m%d_%H%M')}.csv"
        # 엑셀 호환을 위한 utf-8-sig 인코딩
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✨ 수집 완료! '{filename}' 파일이 생성되었습니다.")
        
        # 간단한 요약 출력
        print("-" * 40)
        print(f"총 수집 건수: {len(df)}건")
        print(df[['날짜', '언론사', '제목']].head())
    else:
        print("⚠️ 수집된 데이터가 없습니다.")

# 실행
run_general_scraper()

📅 [2026.02.23]부터 오늘까지의 모든 기사 수집을 시작합니다.


✅ 수집된 목록: 총 0건
🚀 본문 수집 및 개인정보 정제 시작 (Thread: 10)...
⚠️ 수집된 데이터가 없습니다.
